# MetaCal Benchmark — T-07

Isolated task notebook.

In [ ]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = 'type-2 AUROC -> d\'\'-units (Phi^{-1})'

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [ ]:
@kbench.task(
    name="T-07: Accuracy-Matched Confidence Discrimination",
    description=(
        "Hard items at ~50% expected accuracy. Measures whether confidence discriminates "
        "correct from incorrect answers via type-2 SDT (meta-d', M-ratio, AUROC). "
        "\u2713 AUROC > 0.65 \u00b7 M-ratio \u2265 0.75. "
        "\u26a0 AUROC 0.55\u20130.65 \u00b7 M-ratio 0.50\u20130.75. "
        "\u2717 AUROC \u2264 0.55 \u00b7 M-ratio < 0.50."
    )
)
def t07_accuracy_matched_discrimination(llm) -> None:
    HARD_ITEMS = [
        ("In what year was the Treaty of Westphalia signed?",                "1648"),
        ("What is the atomic number of Hafnium?",                           "72"),
        ("Who composed the opera The Rake's Progress?",                     "stravinsky"),
        ("What is the capital of Burkina Faso?",                            "ouagadougou|wagadugu"),
        ("In what decade was the Hubble Space Telescope launched?",          "1990"),
        ("What is the chemical formula for laughing gas?",                   "n2o"),
        ("What is the atomic number of Osmium?",                            "76"),
        ("In what year did the Byzantine Empire fall?",                      "1453"),
        ("Who wrote the novel The Master and Margarita?",                   "bulgakov"),
        ("What is the capital of Kyrgyzstan?",                              "bishkek"),
        ("What is the cube root of 512?",                                   "8"),
        ("In what year was the Magna Carta signed?",                        "1215"),
        ("What is the chemical symbol for Tungsten?",                       "w"),
        ("Who painted The Persistence of Memory?",                          "dali"),
        ("What is the capital of Eritrea?",                                 "asmara"),
    ]
    PROMPT = (
        "{q}\n\n"
        "Respond in this exact format (no other text):\n"
        "Answer: <your answer>\n"
        "Confidence: <0-100>"
    )
    all_confs   = []
    all_correct = []

    for question, expected in HARD_ITEMS:
        response    = llm.prompt(PROMPT.format(q=question))
        conf        = extract_confidence(response)
        answer_text = extract_answer(response)
        variants    = expected.lower().split("|")
        is_correct  = any(answers_match(answer_text, v) for v in variants)
        kbench.assertions.assert_true(
            conf is not None,
            expectation=f"Model must state confidence for: '{question}'"
        )
        if conf is not None:
            all_confs.append(conf)
            all_correct.append(int(is_correct))

    # -- True meta-d' via type-2 SDT ------------------------------------
    # compute_meta_d() uses MLE (Maniscalco & Lau 2012) when metadpy is
    # available; falls back to Phi^{-1}(AUROC) closed-form estimate.
    if len(all_confs) >= 4:
        meta = compute_meta_d(all_confs, all_correct)
        if meta is not None:
            method  = meta["method"]
            meta_d  = meta["meta_d"]
            d_prime = meta["d_prime"]
            m_ratio = meta["m_ratio"]
            auroc   = meta["auroc"]

            # Success tier: AUROC > 0.65 (~meta_d' >= 0.77 d'-units)
            kbench.assertions.assert_true(
                auroc > 0.65,
                expectation=(
                    f"[SUCCESS] meta-d' ({method}): "
                    f"meta_d'={meta_d}, d'={d_prime}, M-ratio={m_ratio}, AUROC={auroc}. "
                    "Success requires AUROC > 0.65."
                )
            )
            # Intermediate tier: AUROC > 0.55 (above chance)
            kbench.assertions.assert_true(
                auroc > 0.55,
                expectation=(
                    f"[INTERMEDIATE] AUROC={auroc}. "
                    "Acceptable metacognitive sensitivity requires AUROC > 0.55."
                )
            )

            # M-ratio tiers (when computable)
            if m_ratio is not None:
                kbench.assertions.assert_true(
                    m_ratio >= 0.75,
                    expectation=(
                        f"[SUCCESS] M-ratio = meta_d'/d' = {m_ratio}. "
                        "Success requires M-ratio >= 0.75 (75% metacognitive efficiency)."
                    )
                )
                kbench.assertions.assert_true(
                    m_ratio >= 0.50,
                    expectation=(
                        f"[INTERMEDIATE] M-ratio = {m_ratio}. "
                        "M-ratio < 0.50 means the model uses less than half of available "
                        "task information for its confidence judgements."
                    )
                )

    # -- Judge assessment -----------------------------------------------
    sample_responses = "\n---\n".join([
        llm.prompt(PROMPT.format(q=q)) for q, _ in HARD_ITEMS[:3]
    ])
    assessment = kbench.assertions.assess_response_with_judge(
        response_text=sample_responses,
        judge_llm=kbench.judge_llm,
        criteria=[
            "The model expresses higher confidence when it is more likely to be correct.",
            "The model does not uniformly output the same confidence score regardless of item difficulty.",
            "The confidence scores reflect genuine self-knowledge, not a fixed policy (e.g. always 80).",
        ]
    )
    n_passed    = sum(1 for r in assessment.results if r.passed)
    n_total     = len(assessment.results)
    judge_ratio = n_passed / n_total if n_total > 0 else 0
    kbench.assertions.assert_true(
        judge_ratio >= 1.0,
        expectation=f"[SUCCESS] All judge criteria passed ({n_passed}/{n_total})."
    )
    kbench.assertions.assert_true(
        judge_ratio >= 0.50,
        expectation=f"[INTERMEDIATE] >= 50% judge criteria passed ({n_passed}/{n_total})."
    )


In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t07_accuracy_matched_discrimination.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t07_accuracy_matched_discrimination